# Glass Classification – Random Forest
**Objective:** Classify glass types using Random Forest and compare Bagging and Boosting.

In [ ]:
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

df=pd.read_excel(r'C:\Users\Admin\OneDrive\Desktop\Data Science ExcelR\Assignments\Assignment 12\glass.xlsx')
print('Shape:',df.shape); display(df.head())

## 1. EDA

In [ ]:
print(df.info())
print('\nMissing values:\n',df.isnull().sum())
print('\nDuplicates:',df.duplicated().sum())
display(df.describe())

df.iloc[:,:-1].hist(figsize=(12,7),bins=20); plt.tight_layout(); plt.show()
sns.boxplot(data=df.iloc[:,:-1]); plt.xticks(rotation=90); plt.show()
sns.heatmap(df.corr(numeric_only=True),annot=True,cmap='coolwarm',fmt='.2f'); plt.show()

## 2. Class Distribution / Imbalance

In [ ]:
print(df['Type'].value_counts().sort_index())
sns.countplot(data=df,x='Type'); plt.title('Glass Type Distribution'); plt.show()

## 3. Preprocessing & Train/Test Split

In [ ]:
X=df.drop(columns='Type'); y=df['Type']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
print(X_train.shape,X_test.shape)

# Standardization is included as requested; it is not essential for tree models.
scaler=StandardScaler()

## 4. Random Forest

In [ ]:
rf=make_pipeline(StandardScaler(),RandomForestClassifier(n_estimators=200,random_state=42,class_weight='balanced'))
rf.fit(X_train,y_train); p=rf.predict(X_test)
print(classification_report(y_test,p,zero_division=0))
print('Accuracy :',round(accuracy_score(y_test,p),4))
print('Precision:',round(precision_score(y_test,p,average='weighted',zero_division=0),4))
print('Recall   :',round(recall_score(y_test,p,average='weighted',zero_division=0),4))
print('F1 Score :',round(f1_score(y_test,p,average='weighted',zero_division=0),4))

## 5. Bagging vs Boosting vs Random Forest

In [ ]:
models={
 'Random Forest':rf,
 'Bagging':make_pipeline(StandardScaler(),BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),n_estimators=100,random_state=42)),
 'Boosting':make_pipeline(StandardScaler(),AdaBoostClassifier(n_estimators=100,random_state=42))
}
results=[]
for name,m in models.items():
    m.fit(X_train,y_train); pr=m.predict(X_test)
    results.append([name,accuracy_score(y_test,pr),precision_score(y_test,pr,average='weighted',zero_division=0),recall_score(y_test,pr,average='weighted',zero_division=0),f1_score(y_test,pr,average='weighted',zero_division=0)])
display(pd.DataFrame(results,columns=['Model','Accuracy','Precision','Recall','F1']))

## 6. Random Forest Feature Importance

In [ ]:
rf_model=rf.named_steps['randomforestclassifier']
imp=pd.Series(rf_model.feature_importances_,index=X.columns).sort_values(ascending=False)
display(imp)
imp.plot(kind='bar',title='Random Forest Feature Importance'); plt.show()

## Conclusion
Random Forest combines many decision trees using bagging and random feature selection. `class_weight='balanced'` helps reduce the effect of class imbalance. Bagging trains models independently on bootstrap samples, while Boosting builds models sequentially and focuses on previous errors. The model comparison shows which ensemble method performs best on the test data.

**Imbalance handling:** use stratified splitting, class weights, oversampling/SMOTE, or undersampling when appropriate. For this assignment, class weights and stratification are used.

## Additional Notes
**Bagging:** Parallel/independent models trained on bootstrap samples; reduces variance.

**Boosting:** Models are trained sequentially, with later models focusing more on previous mistakes; reduces bias and can improve accuracy.

**Main difference:** Bagging mainly reduces variance through averaging; Boosting mainly reduces bias through sequential error correction.